In [3]:
# %pip install -U ultralytics

In [2]:
# Import of all packages
import ultralytics
from ultralytics import YOLO
import shutil
import zipfile
from pathlib import Path
import yaml
from google.colab import drive

drive.mount("/content/drive")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Mounted at /content/drive


In [8]:
# Here are done some configurations and openning of files

DRIVE_ZIP = Path(
    "/content/drive/MyDrive/DLE/taco_yolo.zip" # Same as taco_yolo_initial_split
)
LOCAL_ZIP = Path("/content/taco_yolo.zip")
DATASET_DIR = Path("/content/taco_yolo")

shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)

with zipfile.ZipFile(LOCAL_ZIP) as archive:
    archive.extractall("/content")

DATA_YAML = DATASET_DIR / "data.yaml"

with DATA_YAML.open(encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

dataset_config["path"] = str(DATASET_DIR)

with DATA_YAML.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        dataset_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

In [ ]:
# The configuration of images for first two trainings were with train/val/test of 80/10/10(1200/150/150)
# For further training, I want to try 4-fold cross-validation, doing only train/test of 90/10(1350/150)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

for split in ("train", "test"):
    image_directory = DATASET_DIR / "images" / split
    label_directory = DATASET_DIR / "labels" / split

    images = [
        path
        for path in image_directory.rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(label_directory.rglob("*.txt"))

    print(
        f"{split:5s}: "
        f"{len(images):4d} images, "
        f"{len(labels):4d} label files"
    )

train: 1200 images, 1200 label files
test :  150 images,  150 label files


In [ ]:
# Training of the model with different parameters
DRIVE_RUNS = Path(
    "/content/drive/MyDrive/YOLO/runs"
)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

model = YOLO("yolo26n.pt")

training_results = model.train(
    data=str(DATA_YAML),
    epochs=1,
    imgsz=960,
    batch=8,
    # device=0,
    patience=30,
    optimizer="auto",
    amp=True,
    workers=2,
    cache=False,
    seed=42,
    plots=True,
    save=True,
    save_period=10,
    project=str(DRIVE_RUNS),
    name="taco10_yolo26n",
)

RUN_DIRECTORY = Path(model.trainer.save_dir)
print("Results saved to:", RUN_DIRECTORY)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/taco_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=taco10_yolo26n-3, nbs=64, nms=Fals

In [ ]:
# First training
# epochs=100,
# imgsz=640,

RUN_DIRECTORY = Path(
    "/content/drive/MyDrive/YOLO/runs/taco10_yolo26n"
)

BEST_WEIGHTS = RUN_DIRECTORY / "weights" / "best.pt"

best_model = YOLO(str(BEST_WEIGHTS))

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
)

print("mAP50:", test_metrics.box.map50)
print("mAP50–95:", test_metrics.box.map)

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 189.6±58.6 MB/s, size: 1811.1 KB)
val: Scanning /content/taco_yolo/labels/test/batch_1... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 404.2it/s 0.4s
val: New cache created: /content/taco_yolo/labels/test/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1it/s 9.0s
                   all        150        485      0.298      0.233      0.209      0.149
                 Other         69        159      0.218      0.258      0.154      0.101
                Bottle         39         54      0.511      0.463      0.443      0.347
            Bottle cap         28         32       0.59       0.27      0.295      0.161
                   Can         20         26      0.303      0

In [ ]:
# Second training (we can see a small improvement)
# epochs=50,
# imgsz=960,

RUN_DIRECTORY2 = Path(
    "/content/drive/MyDrive/YOLO/runs/taco10_yolo26n-2"
)

BEST_WEIGHTS2 = RUN_DIRECTORY2 / "weights" / "best.pt"

best_model2 = YOLO(str(BEST_WEIGHTS2))

test_metrics2 = best_model2.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=960,
    device=0,
)

print("mAP50:", test_metrics2.box.map50)
print("mAP50–95:", test_metrics2.box.map)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 174.6±45.2 MB/s, size: 1191.6 KB)
val: Scanning /content/taco_yolo/labels/test/batch_1... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 431.5it/s 0.3s
val: New cache created: /content/taco_yolo/labels/test/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1s/it 10.6s
                   all        150        485      0.359       0.28      0.215      0.154
                 Other         69        159      0.264      0.302      0.182      0.128
                Bottle         39         54      0.481      0.537      0.464      0.333
            Bottle cap         28         32      0.481       0.29      0.359       0.24
                   Can         20         26      0.295      

In [9]:
# Once analysed the previous values, to improve the performance, I decided to apply cross-validation
# That is why I modified the dataset, putting all the validation data together with training.
# It is applied a four-cross validation to try to improve the results

# Here are done some configurations and openning of files

DRIVE_ZIP2 = Path(
    "/content/drive/MyDrive/DLE/taco_yolo_test_train.zip" # Same as taco_yolo_test_train
)
LOCAL_ZIP2 = Path("/content/taco_yolo_test_train.zip")
DATASET_DIR2 = Path("/content/taco_yolo_test_train")

shutil.copy2(DRIVE_ZIP2, LOCAL_ZIP2)

with zipfile.ZipFile(LOCAL_ZIP2) as archive2:
    archive2.extractall("/content")

DATA_YAML2 = DATASET_DIR2 / "data.yaml"

with DATA_YAML2.open(encoding="utf-8") as file2:
    dataset_config2 = yaml.safe_load(file2)

dataset_config2["path"] = str(DATASET_DIR2)

with DATA_YAML2.open("w", encoding="utf-8") as file2:
    yaml.safe_dump(
        dataset_config2,
        file2,
        sort_keys=False,
        allow_unicode=True,
    )

In [7]:
from sklearn.model_selection import KFold

# Only the training portion—not the test images.
TRAIN_IMAGES_DIR = DATASET_DIR2 / "images" / "train"

FOLDS_DIRECTORY = DATASET_DIR2 / "cross_validation"
FOLDS_DIRECTORY.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

training_images = sorted(
    path.resolve()
    for path in TRAIN_IMAGES_DIR.rglob("*")
    if path.suffix.lower() in IMAGE_EXTENSIONS
)

print("Images available for cross-validation:", len(training_images))

with open(DATA_YAML2, encoding="utf-8") as file:
    original_configuration = yaml.safe_load(file)

class_names = original_configuration["names"]

kfold = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42,
)

fold_yamls = []

for fold_number, (train_indices, val_indices) in enumerate(
    kfold.split(training_images),
    start=1,
):
    fold_directory = FOLDS_DIRECTORY / f"fold_{fold_number}"
    fold_directory.mkdir(parents=True, exist_ok=True)

    fold_train_images = [
        training_images[index] for index in train_indices
    ]
    fold_val_images = [
        training_images[index] for index in val_indices
    ]

    train_txt = fold_directory / "train.txt"
    val_txt = fold_directory / "val.txt"

    train_txt.write_text(
        "\n".join(str(path) for path in fold_train_images) + "\n"
    )
    val_txt.write_text(
        "\n".join(str(path) for path in fold_val_images) + "\n"
    )

    fold_yaml = fold_directory / "data.yaml"

    fold_configuration = {
        "path": str(DATASET_DIR2.resolve()),
        "train": str(train_txt.resolve()),
        "val": str(val_txt.resolve()),
        "names": class_names,
    }

    with open(fold_yaml, "w", encoding="utf-8") as file:
        yaml.safe_dump(
            fold_configuration,
            file,
            sort_keys=False,
        )

    fold_yamls.append(fold_yaml)

    print(
        f"Fold {fold_number}: "
        f"{len(fold_train_images)} train, "
        f"{len(fold_val_images)} validation"
    )

Images available for cross-validation: 1350
Fold 1: 1012 train, 338 validation
Fold 2: 1012 train, 338 validation
Fold 3: 1013 train, 337 validation
Fold 4: 1013 train, 337 validation


In [ ]:
# Training of the model with different parameters
DRIVE_RUNS = Path(
    "/content/drive/MyDrive/YOLO/runs"
)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

model = YOLO("yolo26n.pt")

training_results = model.train(
    data=str(DATA_YAML),
    epochs=1,
    imgsz=960,
    batch=8,
    # device=0,
    patience=30,
    optimizer="auto",
    amp=True,
    workers=2,
    cache=False,
    seed=42,
    plots=True,
    save=True,
    save_period=10,
    project=str(DRIVE_RUNS),
    name="taco10_yolo26n",
)

RUN_DIRECTORY = Path(model.trainer.save_dir)
print("Results saved to:", RUN_DIRECTORY)

In [ ]:
DRIVE_RUNS2 = Path(
    "/content/drive/MyDrive/YOLO_2/runs/cross_validation"
)
DRIVE_RUNS2.mkdir(parents=True, exist_ok=True)

fold_results = []

for fold_number, fold_yaml in enumerate(fold_yamls, start=1):
    print(f"\nTraining fold {fold_number}/4")

    # Important: reset to the same pretrained weights for every fold.
    model = YOLO("yolo26n.pt")

    model.train(
        data=str(fold_yaml),
        epochs=50,
        imgsz=960,
        batch=8,
        # device=0,
        patience=30,
        optimizer="auto",
        amp=True,
        workers=2,
        cache=False,
        seed=42,
        deterministic=True,
        plots=True,
        save=True,
        save_period=-1,
        project=str(DRIVE_RUNS2),
        name=f"taco10_yolo26n_fold_{fold_number}",
    )


Training fold 1/4
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/taco_yolo_test_train/cross_validation/fold_1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, mu

In [11]:
# As it couldn't get executed in one execution because time limits in Collab
# it have been executed since last checkpoint

fold4_directory = (
    DRIVE_RUNS2 / "taco10_yolo26n_fold_4"
)

last_weights = (
    fold4_directory / "weights" / "last.pt"
)

print("Checkpoint:", last_weights)
print("Exists:", last_weights.exists())

model4 = YOLO(str(last_weights))
resume_results4 = model4.train(resume=True)

In [17]:
# Here we represent the results of 4 folder

fold_results = []

for fold_number, fold_yaml in enumerate(fold_yamls, start=1):
    best_weights = (
        DRIVE_RUNS2
        / f"taco10_yolo26n_fold_{fold_number}"
        / "weights"
        / "best.pt"
    )

    print(f"Validating fold {fold_number}")
    print("Weights:", best_weights)

    best_model = YOLO(str(best_weights))

    metrics = best_model.val(
        data=str(fold_yaml),
        split="val",
        imgsz=960,
        batch=8,
        workers=2,
        plots=False,
        verbose=False,
    )

    fold_results.append({
        "fold": fold_number,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
    })

cv_results = pd.DataFrame(fold_results)
display(cv_results)

display(
    cv_results[
        ["precision", "recall", "mAP50", "mAP50-95"]
    ].agg(["mean", "std"])
)

Validating fold 1
Weights: /content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_1/weights/best.pt
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 842.3±172.9 MB/s, size: 1745.4 KB)
val: Scanning /content/taco_yolo_test_train/labels/train/batch_1... 338 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 338/338 1.0Kit/s 0.3s
val: New cache created: /content/taco_yolo_test_train/labels/train/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 3.7s/it 2:37
                   all        338       1031      0.405      0.337      0.303      0.236
Speed: 4.4ms preprocess, 388.0ms inference, 0.0ms loss, 0.1ms postprocess per image
Validating fold 2
Weights: /content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_2/we

,fold,precision,recall,mAP50,mAP50-95
0,1,0.404787,0.337383,0.303204,0.235606
1,2,0.381720,0.261322,0.241957,0.183829
2,3,0.357015,0.308492,0.279427,0.213292
3,4,0.347436,0.291117,0.255677,0.197291


,precision,recall,mAP50,mAP50-95
mean,0.372740,0.299578,0.270066,0.207504
std,0.025789,0.031853,0.026975,0.022271


It have been some days since this last execution, so to get some order and make the comparison fair let's add a training with same amount of validation images.

In [5]:
search_directories = [
    Path("/content/drive/MyDrive/YOLO"),
    Path("/content/drive/MyDrive/YOLO_2"),
]

for directory in search_directories:
    if directory.exists():
        for checkpoint in sorted(
            directory.glob("**/weights/best.pt")
        ):
            print(checkpoint)

/content/drive/MyDrive/YOLO/runs/taco10_yolo26n/weights/best.pt
/content/drive/MyDrive/YOLO/runs/taco10_yolo26n-2/weights/best.pt
/content/drive/MyDrive/YOLO/runs/taco10_yolo26n-3/weights/best.pt
/content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_1/weights/best.pt
/content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_2/weights/best.pt
/content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_3/weights/best.pt
/content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_4/weights/best.pt


In [6]:
TRAINING_1_WEIGHTS = Path(
    "/content/drive/MyDrive/YOLO/runs/"
    "taco10_yolo26n/weights/best.pt"
)

TRAINING_2_WEIGHTS = Path(
    "/content/drive/MyDrive/YOLO/runs/"
    "taco10_yolo26n-2/weights/best.pt"
)

CV_RUNS_DIRECTORY = Path(
    "/content/drive/MyDrive/YOLO_2/runs/cross_validation"
)

candidates = [
    {
        "model": "Training 1",
        "group": "single_split",
        "weights": TRAINING_1_WEIGHTS,
    },
    {
        "model": "Training 2",
        "group": "single_split",
        "weights": TRAINING_2_WEIGHTS,
    },
]

for fold_number in range(1, 5):
    candidates.append({
        "model": f"CV Fold {fold_number}",
        "group": "cross_validation",
        "weights": (
            CV_RUNS_DIRECTORY
            / f"taco10_yolo26n_fold_{fold_number}"
            / "weights"
            / "best.pt"
        ),
    })

for candidate in candidates:
    weights = candidate["weights"]
    print(candidate["model"], "->", weights)

    if not weights.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {weights}"
        )

Training 1 -> /content/drive/MyDrive/YOLO/runs/taco10_yolo26n/weights/best.pt
Training 2 -> /content/drive/MyDrive/YOLO/runs/taco10_yolo26n-2/weights/best.pt
CV Fold 1 -> /content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_1/weights/best.pt
CV Fold 2 -> /content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_2/weights/best.pt
CV Fold 3 -> /content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_3/weights/best.pt
CV Fold 4 -> /content/drive/MyDrive/YOLO_2/runs/cross_validation/taco10_yolo26n_fold_4/weights/best.pt


In [10]:
TEST_RESULTS_DIRECTORY = (
    CV_RUNS_DIRECTORY / "common_test_evaluation"
)
TEST_RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

test_results = []

for candidate in candidates:
    print(f"\nTesting {candidate['model']}")

    model = YOLO(str(candidate["weights"]))

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=960,
        batch=8,
        workers=2,
        plots=True,
        verbose=False,
        project=str(TEST_RESULTS_DIRECTORY),
        name=candidate["model"]
            .lower()
            .replace(" ", "_"),
        exist_ok=True,
    )

    test_results.append({
        "model": candidate["model"],
        "group": candidate["group"],
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
    })


Testing Training 1
Ultralytics 8.4.136 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 136.2±26.3 MB/s, size: 1375.6 KB)
val: Scanning /content/taco_yolo/labels/test/batch_1... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 287.6it/s 0.5s
val: New cache created: /content/taco_yolo/labels/test/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.9it/s 10.2s
                   all        150        485       0.31      0.233      0.201       0.15
Speed: 5.3ms preprocess, 15.4ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/drive/MyDrive/YOLO_2/runs/cross_validation/common_test_evaluation/training_1

Testing Training 2
Ultralytics 8.4.136 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (f

NameError: name 'pd' is not defined

In [13]:
test_results_df = pd.DataFrame(test_results)
display(test_results_df)

,model,group,precision,recall,mAP50,mAP50-95
0,Training 1,single_split,0.310048,0.233260,0.201001,0.150287
1,Training 2,single_split,0.357134,0.277988,0.212594,0.152525
2,CV Fold 1,cross_validation,0.373825,0.297314,0.269232,0.186291
3,CV Fold 2,cross_validation,0.346034,0.243695,0.233916,0.173992
4,CV Fold 3,cross_validation,0.246806,0.302375,0.232978,0.168367
5,CV Fold 4,cross_validation,0.362130,0.258383,0.234329,0.177217
